# imports

In [5]:
import sys
sys.path.append(r'\\192.168.10.106\imdea\DataDriven_UT_AlbertoVicente\10_code\UTvsXCT-preprocessing')
from preprocess_tools import onlypores as onlypores_module, io, aligner, reslicer
import numpy as np
from pathlib import Path
from datetime import datetime
import traceback

# volume list

List every XCT volume to process. Each entry is run through the same pipeline as `onlypores.ipynb` (reslice -> find walls -> detect pores -> save outputs + report) in the volume's own folder.

In [6]:
volume_paths = [
    Path(r'/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_10_1/volume_eq_aligned.tif')
]

# processing parameters

In [7]:
sauvola_radius = 15
sauvola_k = 0.2
min_size_filtering = 8

# batch processing

Runs the full onlypores pipeline for every volume in `volume_paths`. Each volume is processed independently: a failure on one volume is logged and skipped rather than stopping the batch.

In [8]:
def outputs_exist(volume_path):
    """Check whether all output files for this volume have already been created."""
    output_dir = volume_path.parent / "onlypores files"
    expected = [
        output_dir / f"{volume_path.stem}_onlypores.tif",
        output_dir / f"{volume_path.stem}_samplemask.tif",
        output_dir / f"{volume_path.stem}_binary.tif",
        output_dir / f"{volume_path.stem}_report.txt",
    ]
    return all(p.exists() for p in expected)


def process_volume(volume_path, sauvola_radius, sauvola_k, min_size_filtering):
    """Run the onlypores pipeline for a single volume and save its outputs + report."""
    print(f"\n{'='*80}\nProcessing: {volume_path}\n{'='*80}")

    volume = io.load_tif(volume_path)
    print(f"Loaded volume shape: {volume.shape}")

    resliced_volume = reslicer.rotate_90(volume, False)
    resliced_volume = reslicer.reslice(resliced_volume, 'Right')
    print(f"Resliced volume shape: {resliced_volume.shape}")

    _, frontwall, backwall = aligner.crop_walls(resliced_volume)
    print(f"Front wall: {frontwall}, back wall: {backwall}")

    pores, sample_mask, binary = onlypores_module.onlypores(
        volume, frontwall, backwall,
        sauvola_radius=sauvola_radius, sauvola_k=sauvola_k,
        min_size_filtering=min_size_filtering
    )

    #save in a "onlypores files" subfolder next to the original volume, with _onlypores, _samplemask and _binary suffixes
    output_dir = volume_path.parent / "onlypores files"
    output_dir.mkdir(parents=True, exist_ok=True)

    io.save_tif(output_dir / f"{volume_path.stem}_onlypores.tif", pores.astype(np.uint8) * 255)
    io.save_tif(output_dir / f"{volume_path.stem}_samplemask.tif", sample_mask.astype(np.uint8) * 255)
    io.save_tif(output_dir / f"{volume_path.stem}_binary.tif", binary.astype(np.uint8) * 255)

    #save a .txt report describing the processing steps and parameters used
    total_voxels = int(np.prod(volume.shape))
    pore_voxels = int(np.sum(pores))
    material_voxels = int(np.sum(sample_mask))
    vvf = pore_voxels / material_voxels if material_voxels > 0 else float('nan')

    report = f"""Pore detection processing report
Generated: {datetime.now().isoformat(timespec='seconds')}

Input volume
------------
Path: {volume_path}
Original shape (Z, Y, X): {volume.shape}

Reslicing
---------
1. rotate_90(volume, clockwise=False)
2. reslice(rotated_volume, 'Right')
Resliced shape: {resliced_volume.shape}

Wall detection (aligner.crop_walls)
------------------------------------
Front wall slice: {frontwall}
Back wall slice: {backwall}
(Slices [0:{frontwall}] and [{backwall}:end] are excluded from pore detection as they belong to the sample walls.)

Pore detection (onlypores.onlypores)
-------------------------------------
1. Volume is cropped to its non-zero bounding box (+2 voxel margin) for efficiency.
2. Sauvola adaptive thresholding is applied slice-by-slice (per Y slice, over the Z-X plane):
   - window_size (sauvola_radius): {sauvola_radius}
   - k: {sauvola_k}
   - r (dynamic range): 128 (fixed)
   - Voxels above the local adaptive threshold are classified as material; below as pore/background.
3. Front wall slices [0:{frontwall}] and back wall slices [{backwall}:end] are forced to material (True) to exclude them from pore detection.
4. A material (sample) mask is generated independently via:
   - Global Otsu thresholding on the cropped volume.
   - Maximum-intensity projection along Z to find the sample's 2D footprint (largest connected component).
   - Internal voids filled (fill_voids) within that footprint, across all Z slices, to obtain a solid sample envelope.
5. Pores = (NOT binary) AND sample_mask, i.e. voxels below the Sauvola threshold that fall inside the solid sample envelope.

Pore cleaning (onlypores.clean_pores)
----------------------------------------
Applied because min_size_filtering > 0.
   - min_size: {min_size_filtering} voxels (3D connected components, 26-connectivity, below this are discarded as noise)
   - Dimensional filter: components must span >= 2 voxels in each of Z, Y, X (removes flat/linear artifacts)

Results
-------
Total volume voxels: {total_voxels}
Material (sample_mask) voxels: {material_voxels}
Pore voxels (after cleaning): {pore_voxels}
Void volumetric fraction (pore voxels / material voxels): {vvf:.6f} ({vvf*100:.4f}%)

Output files (in this folder)
------------------------------
{volume_path.stem}_onlypores.tif   - binary pore mask (255 = pore, 0 = not pore)
{volume_path.stem}_samplemask.tif  - binary sample/material envelope mask (255 = material, 0 = background)
{volume_path.stem}_binary.tif      - raw Sauvola thresholding result before pore/mask combination (255 = material, 0 = background)
"""

    report_path = output_dir / f"{volume_path.stem}_report.txt"
    report_path.write_text(report, encoding="utf-8")
    print(f"Saved outputs and report to {output_dir}")

    return {"volume_path": volume_path, "vvf": vvf, "pore_voxels": pore_voxels, "material_voxels": material_voxels}


results = []
errors = []
skipped = []

for volume_path in volume_paths:
    if outputs_exist(volume_path):
        print(f"SKIP (already processed): {volume_path}")
        skipped.append(volume_path)
        continue
    try:
        result = process_volume(volume_path, sauvola_radius, sauvola_k, min_size_filtering)
        results.append(result)
    except Exception as e:
        print(f"ERROR processing {volume_path}: {e}")
        traceback.print_exc()
        errors.append((volume_path, e))

print(f"\n{'='*80}\nBatch complete: {len(results)} succeeded, {len(errors)} failed, {len(skipped)} skipped (already processed)\n{'='*80}")
for r in results:
    print(f"  OK      {r['volume_path']}  VVF={r['vvf']*100:.4f}%")
for volume_path, e in errors:
    print(f"  FAIL    {volume_path}  ({e})")
for volume_path in skipped:
    print(f"  SKIPPED {volume_path}")


Processing: /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_10_1/volume_eq_aligned.tif
Loaded volume shape: (243, 3849, 1941)
Resliced volume shape: (3849, 1941, 243)
Computing material mask using sequential processing...
Filling internal voids...
Material mask generation complete.
Front wall: 12, back wall: 233
Starting pore detection analysis...
Computing volume bounding box...
Original volume shape: (243, 3849, 1941)
Data bounding box: Z[0:242], Y[0:3848], X[0:1940]
Cropped volume shape: (243, 3849, 1941)
Applying Sauvola adaptive thresholding...
Volume size: 1.69 GB
Available memory: 171.35 GB
Required memory estimate: 3.38 GB
Using parallel implementation...
Applying Sauvola thresholding with parallel processing...
Excluding front wall: slices 0 to 11
Excluding back wall: slices 233 to end
Generating material mask...
Computing material mask using sequential processing...
Filling internal voids...
Material mas

/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/10_code/UTvsXCT-preprocessing/preprocess_tools/onlypores.py:512: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  labeled_pores = remove_small_objects(labeled_pores, min_size=min_size, connectivity=3)


  Components after size filtering: 182290
  Valid components after dimensional filtering: 177238
  Components after dimensional filtering: 177238
  Total pore voxels retained: 12937670
After filtering: 12937670 pore voxels remaining.
Pore detection analysis complete.
Saved outputs and report to /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_10_1/onlypores files

Batch complete: 1 succeeded, 0 failed, 0 skipped (already processed)
  OK      /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_10_1/volume_eq_aligned.tif  VVF=0.7622%
